In [3]:
from finn.util.basic import make_build_dir
from finn.util.visualization import showInNetron
import os

import torch
import onnx
from finn.util.test import get_test_model_trained
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
import finn.transformation.streamline.absorb as absorb
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.transformation.streamline import Streamline
from qonnx.transformation.general import RemoveUnusedTensors
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs

from model import UNetQ  # Import your model definition

# Clean-up
import os
import glob

# Get all .onnx files in the current folder
onnx_files = glob.glob("*.onnx")

# Delete each file
for file in onnx_files:
    os.remove(file)
    print(f"Deleted: {file}")


build_dir = "./" #os.environ["FINN_BUILD_DIR"]  # Typically /workspace inside Docker
MODEL_WEIGHTS = os.path.join(build_dir, "best_unet_weights.pth")  # /workspace/best_unet_brevitas_weights.pth
unetm = UNetQ(in_channels=1, out_channels=1)
state_dict = torch.load(MODEL_WEIGHTS, map_location="cpu")
unetm.load_state_dict(state_dict)
export_onnx_path = build_dir + "/unet_export.onnx"
export_qonnx(unetm, torch.randn(1, 1, 128, 128), export_onnx_path)

qonnx_cleanup(export_onnx_path, out_file=export_onnx_path)

# Load and convert to FINN
model = ModelWrapper("unet_export.onnx")
model = model.transform(ConvertQONNXtoFINN())
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(Streamline())
model = model.transform(LowerConvsToMatMul())
model = model.transform(ConvertBipolarMatMulToXnorPopcount())
model = model.transform(Streamline())
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())

model.save(build_dir + "/unet_q_streamlined.onnx")

Deleted: unet_q_streamlined.onnx
Deleted: unet_export.onnx
Deleted: QuantSimpleUnet_512x512x1_1classes_1batch_4W4A_noSkip_upsample_qonnx-hw_layers.onnx
Deleted: unet_q_streamlined_inferupsample.onnx


/home/komaro/デスクトップ/Cermak/finn/deps/qonnx/src/qonnx/core/onnx_exec.py:101: UserWarning: Output shapes disagree after node input: "/Sub_2_output_0"
output: "/ConstantOfShape_output_0"
name: "/ConstantOfShape"
op_type: "ConstantOfShape"
attribute {
  name: "value"
  t {
    dims: 1
    data_type: 7
    raw_data: "\000\000\000\000\000\000\000\000"
  }
  type: TENSOR
}
 execution:
                    found (4,) vs expected (0,)
  warnings.warn(
/home/komaro/デスクトップ/Cermak/finn/deps/qonnx/src/qonnx/core/onnx_exec.py:101: UserWarning: Output shapes disagree after node input: "/Cast_output_0"
input: "/ConstantOfShape_output_0"
output: "/Concat_2_output_0"
name: "/Concat_2"
op_type: "Concat"
attribute {
  name: "axis"
  i: 0
  type: INT
}
 execution:
                    found (8,) vs expected (0,)
  warnings.warn(
/home/komaro/デスクトップ/Cermak/finn/deps/qonnx/src/qonnx/core/onnx_exec.py:101: UserWarning: Output shapes disagree after node input: "/Concat_2_output_0"
input: "/Constant_10_output_0"


In [4]:
import onnx

def print_onnx_graph_nodes(onnx_path):
    # Load the ONNX model
    model = onnx.load(onnx_path)
    
    print("=== ONNX Nodes ===")
    for node in model.graph.node:
        print(f"Node name: {node.name}, op_type: {node.op_type}")
        print(f"  inputs : {node.input}")
        print(f"  outputs: {node.output}")
        print()

def print_onnx_tensors(onnx_path):
    model = onnx.load(onnx_path)

    print("=== Model Inputs ===")
    for inp in model.graph.input:
        print(f"Input name: {inp.name}")
        # shape
        if inp.type.tensor_type.shape.dim:
            dims = [dim.dim_value for dim in inp.type.tensor_type.shape.dim]
            print(f"  shape: {dims}")
        # data type
        dtype = inp.type.tensor_type.elem_type
        print(f"  data type: {dtype}")
        print()

    print("=== Model Outputs ===")
    for out in model.graph.output:
        print(f"Output name: {out.name}")
        # shape
        if out.type.tensor_type.shape.dim:
            dims = [dim.dim_value for dim in out.type.tensor_type.shape.dim]
            print(f"  shape: {dims}")
        # data type
        dtype = out.type.tensor_type.elem_type
        print(f"  data type: {dtype}")
        print()

    print("=== Model Initializers ===")
    for init in model.graph.initializer:
        print(f"Initializer name: {init.name}")
        print(f"  dims: {init.dims}")
        print(f"  data_type: {init.data_type}")
        # If you want the raw data, be aware it could be large:
        # raw_data = init.raw_data
        print()


In [5]:
print_onnx_graph_nodes(build_dir +  "/unet_q_streamlined.onnx")
print_onnx_tensors(build_dir + "/unet_q_streamlined.onnx")
showInNetron(build_dir + "/unet_q_streamlined.onnx")

=== ONNX Nodes ===
Node name: MultiThreshold_0, op_type: MultiThreshold
  inputs : ['global_in', 'MultiThreshold_0_param0']
  outputs: ['MultiThreshold_0_out0']

Node name: Transpose_0, op_type: Transpose
  inputs : ['MultiThreshold_0_out0']
  outputs: ['Transpose_0_out0']

Node name: Im2Col_0, op_type: Im2Col
  inputs : ['Transpose_0_out0']
  outputs: ['Im2Col_0_out0']

Node name: MatMul_0, op_type: MatMul
  inputs : ['Im2Col_0_out0', 'MatMul_0_param0']
  outputs: ['MatMul_0_out0']

Node name: MultiThreshold_1, op_type: MultiThreshold
  inputs : ['MatMul_0_out0', 'MultiThreshold_1_param0']
  outputs: ['DZxHPb']

Node name: , op_type: Transpose
  inputs : ['DZxHPb']
  outputs: ['MultiThreshold_1_out0']

Node name: MaxPool_0, op_type: MaxPool
  inputs : ['MultiThreshold_1_out0']
  outputs: ['MaxPool_0_out0']

Node name: Transpose_2, op_type: Transpose
  inputs : ['MaxPool_0_out0']
  outputs: ['Transpose_2_out0']

Node name: Im2Col_1, op_type: Im2Col
  inputs : ['Transpose_2_out0']
  out

In [6]:
model = ModelWrapper("unet_q_streamlined.onnx")

# let's see all the tensors in the model
all_tensors = model.get_all_tensor_names()

for tname in all_tensors:
    dtype = model.get_tensor_datatype(tname)
    print(f"Tensor: {tname}, DataType: {dtype}")

    # find all Im2Col nodes and print their input datatypes
im2col_nodes = model.get_nodes_by_op_type("Im2Col")
for idx, node in enumerate(im2col_nodes):
    in_name = node.input[0]  # the name of the tensor feeding into Im2Col
    dt = model.get_tensor_datatype(in_name)
    print(f"Im2Col_{idx} input tensor name: {in_name}, data type: {dt}")

Tensor: Mul_0_out0, DataType: FLOAT32
Tensor: Pad_0_param0, DataType: FLOAT32
Tensor: MultiThreshold_0_param0, DataType: FLOAT32
Tensor: MultiThreshold_1_out0, DataType: INT8
Tensor: MultiThreshold_2_out0, DataType: INT8
Tensor: MultiThreshold_3_out0, DataType: INT8
Tensor: MultiThreshold_4_out0, DataType: INT8
Tensor: MultiThreshold_0_out0, DataType: INT8
Tensor: MaxPool_0_out0, DataType: INT8
Tensor: MaxPool_1_out0, DataType: INT8
Tensor: MaxPool_2_out0, DataType: INT8
Tensor: MaxPool_3_out0, DataType: INT8
Tensor: Transpose_9_out0, DataType: INT32
Tensor: Mul_0_param0, DataType: FLOAT32
Tensor: MultiThreshold_1_param0, DataType: INT32
Tensor: MultiThreshold_2_param0, DataType: INT32
Tensor: MultiThreshold_3_param0, DataType: INT32
Tensor: MultiThreshold_4_param0, DataType: INT32
Tensor: MatMul_0_param0, DataType: INT8
Tensor: Transpose_0_out0, DataType: INT8
Tensor: Im2Col_0_out0, DataType: INT8
Tensor: MatMul_0_out0, DataType: INT32
Tensor: MatMul_1_param0, DataType: INT8
Tensor: T

In [16]:
from finn.transformation.fpgadataflow.convert_to_hw_layers import InferThresholdingLayer
import finn.transformation.streamline.absorb as absorb
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import RemoveUnusedTensors
from qonnx.core.modelwrapper import ModelWrapper

# Load the ONNX model
model = ModelWrapper("unet_q_streamlined.onnx")

# Debug: Print graph nodes before transformations
print("Graph nodes before transformations:")
for node in model.graph.node:
    print(f"Node: {node.name}, OpType: {node.op_type}")

# Apply transformations
# 1. Absorb scalar multiplication and addition into top layers
model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())

# 2. Absorb consecutive Transpose nodes
model = model.transform(absorb.AbsorbConsecutiveTransposes())

# 3. Infer thresholding layers for hardware compatibility
model = model.transform(InferThresholdingLayer())

# 4. Infer data types to ensure proper quantization and remove floats
model = model.transform(InferDataTypes())

# 5. Remove Pad nodes explicitly
new_graph = []
for node in model.graph.node:
    if node.op_type != "Pad":
        new_graph.append(node)
model.graph.ClearField("node")
model.graph.node.extend(new_graph)

# 6. Remove unused tensors to clean up the graph
model = model.transform(RemoveUnusedTensors())

# Debug: Print graph nodes after transformations
print("\nGraph nodes after transformations:")
for node in model.graph.node:
    print(f"Node: {node.name}, OpType: {node.op_type}")

# Save the updated model
model.save("unet_transformed_no_pad.onnx")
print("\nTransformed model without Pad node saved as 'unet_transformed_no_pad.onnx'")


for node in model.graph.node:
    print(f"Node: {node.name}, Inputs: {node.input}, Outputs: {node.output}")


Graph nodes before transformations:
Node: MultiThreshold_0, OpType: MultiThreshold
Node: Transpose_0, OpType: Transpose
Node: Im2Col_0, OpType: Im2Col
Node: MatMul_0, OpType: MatMul
Node: MultiThreshold_1, OpType: MultiThreshold
Node: , OpType: Transpose
Node: MaxPool_0, OpType: MaxPool
Node: Transpose_2, OpType: Transpose
Node: Im2Col_1, OpType: Im2Col
Node: MatMul_1, OpType: MatMul
Node: MultiThreshold_2, OpType: MultiThreshold
Node: , OpType: Transpose
Node: MaxPool_1, OpType: MaxPool
Node: Transpose_4, OpType: Transpose
Node: Im2Col_2, OpType: Im2Col
Node: MatMul_2, OpType: MatMul
Node: MultiThreshold_3, OpType: MultiThreshold
Node: , OpType: Transpose
Node: MaxPool_2, OpType: MaxPool
Node: Transpose_6, OpType: Transpose
Node: Im2Col_3, OpType: Im2Col
Node: MatMul_3, OpType: MatMul
Node: MultiThreshold_4, OpType: MultiThreshold
Node: , OpType: Transpose
Node: MaxPool_3, OpType: MaxPool
Node: Transpose_8, OpType: Transpose
Node: MatMul_4, OpType: MatMul
Node: Transpose_9, OpType: Tr

In [17]:
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition
from finn.transformation.streamline.absorb import AbsorbConsecutiveTransposes, AbsorbScalarMulAddIntoTopK
from finn.transformation.fpgadataflow.insert_tlastmarker import InsertTLastMarker
from qonnx.transformation.general import RemoveUnusedTensors
from qonnx.transformation.infer_datatypes import InferDataTypes
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
from finn.transformation.fpgadataflow.create_generic_partitions import PartitionFromLambda
from finn.core.modelwrapper import ModelWrapper

# Load the ONNX model
model_path = "unet_transformed_no_pad.onnx"
output_path = "unet_final_hw_compatible.onnx"
model = ModelWrapper(model_path)

# Debug: Print initial nodes
print("\nGraph nodes before transformations:")
for node in model.graph.node:
    print(f"Node: {node.name}, OpType: {node.op_type}")

# Step 1: Absorb consecutive transposes
print("\nApplying AbsorbConsecutiveTransposes...")
model = model.transform(AbsorbConsecutiveTransposes())

# Step 2: Absorb scalar multiplications/additions into neighboring nodes
print("\nApplying AbsorbScalarMulAddIntoTopK...")
model = model.transform(AbsorbScalarMulAddIntoTopK())

# Step 3: Infer integer data types to remove floating-point dependencies
print("\nApplying InferDataTypes...")
model = model.transform(InferDataTypes())

# Step 4: Move scalar linear operations past invariants
print("\nApplying MoveScalarLinearPastInvariants...")
model = model.transform(MoveScalarLinearPastInvariants())

# Step 5: Remove unused tensors to clean up the graph
print("\nApplying RemoveUnusedTensors...")
model = model.transform(RemoveUnusedTensors())

# Step 6: Create a cycle-free partition to avoid node self-dependencies
def partitioning(node):
    # Example: Assign even/odd partitions based on node name hash
    return hash(node.name) % 2

print("\nApplying PartitionFromLambda...")
model = model.transform(PartitionFromLambda(partitioning=partitioning))

# Step 7: Insert TLastMarker for hardware compatibility
print("\nApplying InsertTLastMarker...")
model = model.transform(InsertTLastMarker())

# Step 8: Apply CreateDataflowPartition for hardware layers
print("\nApplying CreateDataflowPartition...")
model = model.transform(CreateDataflowPartition())

# Save the final model
print("\nSaving final transformed model...")
model.save(output_path)

# Debug: Print final nodes
print("\nGraph nodes after transformations:")
for node in model.graph.node:
    print(f"Node: {node.name}, OpType: {node.op_type}")

print(f"\nFinal transformed model saved as '{output_path}'")


ModuleNotFoundError: No module named 'finn.transformation.fpgadataflow.create_generic_partitions'

In [15]:
from finn.transformation.streamline import Streamline
from finn.transformation.fpgadataflow.convert_to_hw_layers import (
    InferStreamingMaxPool,
    InferThresholdingLayer,
    InferQuantizedMatrixVectorActivation,
)
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition
from finn.builder.build_dataflow_steps import build_dataflow_step_lookup
from qonnx.transformation.infer_data_layouts import InferDataLayouts

# Perform additional streamlining
model = model.transform(Streamline())

# Absorb remaining transposes into neighboring nodes
model = model.transform(absorb.AbsorbConsecutiveTransposes())

# Infer streaming-compatible hardware layers
model = model.transform(InferStreamingMaxPool())
model = model.transform(InferThresholdingLayer())
model = model.transform(InferQuantizedMatrixVectorActivation())

# Infer tensor layouts
model = model.transform(InferDataLayouts())

# Create hardware-compatible partitions
model = model.transform(CreateDataflowPartition())

# Save the final transformed model
model.save("unet_final_hw_compatible.onnx")


AssertionError: cycle-free graph violated: partition depends on itself

In [7]:
from finn.util.basic import pynq_part_map
# change this if you have a different PYNQ board, see list above
pynq_board = "Pynq-Z2"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10

import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.infer_data_layouts import InferDataLayouts

model = ModelWrapper(build_dir + "/unet_q_streamlined.onnx")
model = model.transform(to_hw.InferBinaryMatrixVectorActivation())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
model = model.transform(to_hw.InferLabelSelectLayer())
model = model.transform(to_hw.InferThresholdingLayer())
model = model.transform(to_hw.InferUpsample())

model.save(build_dir + "/unet_q_streamlined_inferupsample.onnx")
showInNetron(build_dir + "/unet_q_streamlined_inferupsample.onnx")

Stopping http://0.0.0.0:8081
Serving './/unet_q_streamlined_inferupsample.onnx' at http://0.0.0.0:8081


In [8]:
from finn.util.basic import pynq_part_map
# change this if you have a different PYNQ board, see list above
pynq_board = "Pynq-Z2"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10

import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.infer_data_layouts import InferDataLayouts

model = ModelWrapper(build_dir + "/unet_q_streamlined_inferupsample.onnx")
model = model.transform(to_hw.InferBinaryMatrixVectorActivation())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
# TopK to LabelSelect
model = model.transform(to_hw.InferLabelSelectLayer())
# input quantization (if any) to standalone thresholding
model = model.transform(to_hw.InferThresholdingLayer())
model = model.transform(to_hw.InferConvInpGen())
model = model.transform(to_hw.InferStreamingMaxPool())
# get rid of Reshape(-1, 1) operation between hw nodes
model = model.transform(RemoveCNVtoFCFlatten())
# get rid of Tranpose -> Tranpose identity seq
model = model.transform(absorb.AbsorbConsecutiveTransposes())
# infer tensor data layouts
model = model.transform(InferDataLayouts())
parent_model = model.transform(CreateDataflowPartition())
parent_model.save(build_dir + "/unet_dataflow_parent.onnx")
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")
# save the dataflow partition with a different name for easier access
# and specialize the layers to HLS variants
dataflow_model = ModelWrapper(dataflow_model_filename)
dataflow_model = dataflow_model.transform(SpecializeLayers(fpga_part))
dataflow_model.save(build_dir + "/unet_dataflow_model.onnx")

AssertionError: cycle-free graph violated: partition depends on itself

In [ ]:
showInNetron(build_dir + "/unet_dataflow_parent.onnx")

In [ ]:
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
model = ModelWrapper(build_dir+"/unet_dataflow_model.onnx")
model = model.transform(ZynqBuild(platform = pynq_board, period_ns = target_clk_ns))

from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver
model = model.transform(MakePYNQDriver("zynq-iodma"))

model.save(build_dir + "/unet_dataflow_synth.onnx")

In [ ]:
from shutil import copy
from distutils.dir_util import copy_tree

# create directory for deployment files
deployment_dir = make_build_dir(prefix="pynq_deployment_")
model.set_metadata_prop("pynq_deployment_dir", deployment_dir)

# get and copy necessary files
# .bit and .hwh file
bitfile = model.get_metadata_prop("bitfile")
hwh_file = model.get_metadata_prop("hw_handoff")
deploy_files = [bitfile, hwh_file]

for dfile in deploy_files:
    if dfile is not None:
        copy(dfile, deployment_dir)

# driver.py and python libraries
pynq_driver_dir = model.get_metadata_prop("pynq_driver_dir")
copy_tree(pynq_driver_dir, deployment_dir)

In [ ]:
import importlib_resources
import matplotlib.pyplot as plt
import numpy as np

ref = importlib_resources.files("finn.qnn-data") / "cifar10/cifar10-test-data-class3.npz"
with importlib_resources.as_file(ref) as fn:
    x = np.load(fn)["arr_0"]
x = x.reshape(3, 32,32).transpose(1, 2, 0)
plt.imshow(x)

model = ModelWrapper(build_dir + "/end2end_cnv_w1a1_synth.onnx")
iname = model.graph.input[0].name
ishape = model.get_tensor_shape(iname)
np.save(deployment_dir + "/input.npy", x.reshape(ishape))

! ls {deployment_dir}

from shutil import make_archive
make_archive('deploy-on-pynq-cnv', 'zip', deployment_dir)